# Лабораторная работа №5


Создать сеть на базе LSTM используя TensorFlow (Keras). Сеть должна принимать на вход текстовый файл и на его базе генерировать свою абракадабру. Отчет должен содержать кроме кода, обучающий файл и результат генерации.

In [99]:
# Библиотеки
import re
import numpy as np
import tensorflow as tf

In [105]:
# Загружаем текст
with open("C:\\Users\\Kirill\\YandexDisk\\Учёба\\7 семестр\\ИИ\\Lab5\\Text.txt", "r") as file:
    text = file.read().lower()

text = re.sub(r"[^а-яa-z0-9\s]", "", text)

In [108]:
chars_count = len(text)
words = text.split()
word_count = len(words)
unique_words = len(set(words))

print(f'Статистика текста:')
print(f'\tКоличество символов: {chars_count}')
print(f'\tКоличество слов: {word_count}')
print(f'\tУникальных слов: {unique_words}')
print(f'\tСредняя длина слова: {chars_count/word_count:.2f} символов')

Статистика текста:
	Количество символов: 236898
	Количество слов: 37859
	Уникальных слов: 11626
	Средняя длина слова: 6.26 символов


In [ ]:
# Производим токинецацию теста 
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts([text])

word_index = tokenizer.word_index
index_word = {i: w for w, i in word_index.items()}
vocab_size = unique_words + 1

In [111]:
# Преобразуем текст в последовательность слов
tokens = tokenizer.texts_to_sequences([text])[0]

In [120]:
# Создаём обучающую последовательность
seq_length = 10
sequences = []

for i in range(seq_length, len(tokens)):
    seq = tokens[i-seq_length:i+1]
    sequences.append(seq)

sequences = np.array(sequences)

X = sequences[:, :-1]
y = sequences[:, -1]


In [131]:
# Создаем архитектуру LSTM
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64),  # Преобразует целочисленные токены в плотные векторы


    tf.keras.layers.LSTM(
        128,                    # Количество LSTM-нейронов
        return_sequences=True,  # Возвращает полную последовательность
        dropout=0.3,            # Забывание нейронов на feed-forward этапе, регуляризация для борьбы с переобучением
        recurrent_dropout=0.3   # Забывание на recurrent этапе
    ),
    
    tf.keras.layers.LSTM(
        128,                  
        dropout=0.3,            
        recurrent_dropout=0.3   
    ),

    tf.keras.layers.Dense(vocab_size, activation="softmax")
])

# Компиляция модели
model.compile(
    loss="sparse_categorical_crossentropy",  # Функция потерь для многоклассовой классификации
    optimizer="adam",                        # Адаптивный оптимизатор 
)

# Обучение модели
history = model.fit(
    X, y,            
    epochs=30,                   # Максимальное количество эпох
    batch_size=64,              # Размер батча (сколько примеров за одну итерацию)
    verbose=1                    # Подробный вывод прогресса
)

Epoch 1/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 29s 38ms/step - loss: 8.6147
Epoch 2/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - loss: 8.1206
Epoch 3/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 37ms/step - loss: 7.9289
Epoch 4/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - loss: 7.7958
Epoch 5/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 37ms/step - loss: 7.7076
Epoch 6/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - loss: 7.6128
Epoch 7/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 37ms/step - loss: 7.5156
Epoch 8/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 22s 38ms/step - loss: 7.4270
Epoch 9/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - loss: 7.3860
Epoch 10/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 37ms/step - loss: 7.2540
Epoch 11/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - loss: 7.1017
Epoch 12/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 37ms/step - loss: 6.9698
Epoch 13/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - loss: 6.8651
Epoch 14/30
571/571 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - loss: 6.7067
Epoch 15/30
571/571 ━━━━━━━━━

In [137]:
model.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_13 (Embedding)        │ (None, 10, 64)         │       797,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_20 (LSTM)                  │ (None, 10, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_21 (LSTM)                  │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 12461)          │     1,607,469 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,906,121 (30.16 MB)

 Trainable params: 2,635,373 (10.05 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,270,748 (20.11 MB)

In [138]:
# Генерируем текст
def generate_text(seed_text, next_words=30):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = token_list[-seq_length:]
        token_list = tf.keras.preprocessing.sequence.pad_sequences(
            [token_list], maxlen=seq_length
        )

        predicted = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted)
        seed_text += " " + index_word.get(predicted_word_index, "")

    return seed_text

In [139]:
print(generate_text("я люблю", 40))

я люблю из держат выводивший и шаги как потом спрятал пугало мамалыга смелая малым роскоши я являюсь добывать не посмел франкенштейн или современный губ столь рассудка и надев за в женеву льняном головокружительного шелли поэтов или современный осветив гвоздями и связной и затемно памятуя однако


In [140]:
print(generate_text("дракула сказал", 20))

дракула сказал и кинулся и взвешенные вещи каждую подносит вспоминать я вздрогнул я не пути я не видом гвозди взбираясь в серыми


In [148]:
print(generate_text("в один день", 10))

в один день если худшее мне потом не  почему и боковую и поступил
